# Search Query Behavior Analysis

## Project Overview
Analyze on-site search logs to understand popular queries, failed searches, zero-result terms, and user intent patterns.

## Learning Objectives
- Analyze search query frequency and long-tail distributions
- Identify zero-result queries that indicate content gaps
- Classify search intent (navigational, informational, transactional)
- Measure search effectiveness via click-through and refinement rates
- Detect trending and declining search terms

## Problem Statement
An e-commerce site's search bar is the second-most-used navigation path. Understanding what users search for — especially when they get no results — reveals content gaps, product demand, and UX issues that directly impact revenue.

## Why This Project Matters
On-site search users convert 2-3× higher than browsers. Failed searches mean lost revenue. Query analysis uncovers what customers want but can't find, guiding product catalog, content, and UX decisions.

## Dataset Overview
Synthetic search log: ~15,000 queries over 3 months with result counts, click-through data, device info, and query refinement flags.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 15000

# Query corpus with popularity weights
queries_pool = {
    'laptop': 500, 'iphone case': 350, 'wireless headphones': 300,
    'running shoes': 280, 'yoga mat': 200, 'bluetooth speaker': 180,
    'usb cable': 170, 'backpack': 160, 'water bottle': 150, 'desk lamp': 140,
    'mouse pad': 130, 'keyboard': 120, 'monitor stand': 110, 'webcam': 100,
    'phone charger': 250, 'tablet': 90, 'smart watch': 200, 'air purifier': 80,
    'coffee maker': 70, 'standing desk': 60,
    # Zero-result / misspelled queries
    'labtop': 40, 'iphoen': 30, 'wireles earbuds': 25, 'blutooth': 20,
    'nike air max 2025': 15, 'product xyz': 10, 'asdfgh': 5,
    # Long-tail
    'ergonomic split keyboard': 12, 'noise cancelling headphones under 50': 18,
    'best laptop for programming 2024': 8, 'cat bed heated': 6,
}

q_names = list(queries_pool.keys())
q_weights = np.array(list(queries_pool.values()), dtype=float)
q_weights /= q_weights.sum()

queries = np.random.choice(q_names, size=n, p=q_weights)
dates = pd.date_range('2024-01-01', periods=90, freq='D')
search_dates = np.random.choice(dates, size=n)
device = np.random.choice(['desktop', 'mobile', 'tablet'], size=n, p=[0.40, 0.50, 0.10])

# Zero-result queries
zero_result_terms = {'labtop', 'iphoen', 'wireles earbuds', 'blutooth',
                     'nike air max 2025', 'product xyz', 'asdfgh'}

results_count = []
clicked = []
refined = []
for q in queries:
    if q in zero_result_terms:
        results_count.append(0)
        clicked.append(0)
        refined.append(int(np.random.random() < 0.4))
    else:
        rc = int(np.random.lognormal(3, 1))
        rc = min(rc, 500)
        results_count.append(rc)
        clicked.append(int(np.random.random() < 0.55))
        refined.append(int(np.random.random() < 0.15))

# Intent classification
intent_map = {
    'laptop': 'transactional', 'iphone case': 'transactional',
    'wireless headphones': 'transactional', 'running shoes': 'transactional',
    'best laptop for programming 2024': 'informational',
    'noise cancelling headphones under 50': 'informational',
    'nike air max 2025': 'navigational', 'usb cable': 'transactional',
}

df = pd.DataFrame({
    'date': search_dates, 'query': queries, 'device': device,
    'results_count': results_count, 'clicked': clicked, 'refined': refined
})
df['zero_result'] = (df['results_count'] == 0).astype(int)
df['intent'] = df['query'].map(lambda q: intent_map.get(q, 'transactional'))
df['month'] = df['date'].dt.to_period('M')
df['day_of_week'] = df['date'].dt.day_name()

print(f'Dataset: {df.shape}')
print(f'Unique queries: {df["query"].nunique()}')
print(f'Zero-result rate: {df["zero_result"].mean():.3f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'\nDevice distribution:\n{df["device"].value_counts()}')
print(f'\nZero-result queries: {df["zero_result"].sum()} / {len(df)} ({df["zero_result"].mean()*100:.1f}%)')

## Top Queries and Long-Tail Distribution

In [ ]:
query_counts = df['query'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 15 queries
query_counts.head(15).plot.barh(ax=axes[0], edgecolor='black')
axes[0].set_title('Top 15 Search Queries')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# Long-tail distribution
axes[1].plot(range(len(query_counts)), query_counts.values, color='steelblue')
axes[1].set_title('Query Frequency Distribution (Long Tail)')
axes[1].set_xlabel('Query Rank')
axes[1].set_ylabel('Frequency')
axes[1].set_yscale('log')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'query_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

# Coverage stats
top10_share = query_counts.head(10).sum() / len(df) * 100
print(f'Top 10 queries account for {top10_share:.1f}% of all searches')

## Zero-Result Query Analysis

In [ ]:
zero_df = df[df['zero_result'] == 1]
zero_counts = zero_df['query'].value_counts()
print('Zero-result queries (content gaps):')
print(zero_counts)

fig, ax = plt.subplots(figsize=(10, 4))
zero_counts.plot.bar(ax=ax, color='salmon', edgecolor='black')
ax.set_title('Zero-Result Queries — Content Gaps')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'zero_result_queries.png'), dpi=100, bbox_inches='tight')
plt.show()

## Click-Through Rate Analysis

In [ ]:
# CTR by query (non-zero results only)
non_zero = df[df['zero_result'] == 0]
ctr_by_query = non_zero.groupby('query').agg(
    searches=('clicked', 'count'),
    clicks=('clicked', 'sum')
)
ctr_by_query['ctr'] = (ctr_by_query['clicks'] / ctr_by_query['searches']).round(3)
ctr_by_query = ctr_by_query.sort_values('searches', ascending=False)
print('CTR for top queries:')
print(ctr_by_query.head(15))

# CTR by device
ctr_device = non_zero.groupby('device')['clicked'].mean()
print(f'\nCTR by device:\n{ctr_device.round(3)}')

## Query Refinement Analysis

In [ ]:
refine_rate = df.groupby('zero_result')['refined'].mean()
print('Refinement rate:')
print(f'  Zero-result queries: {refine_rate[1]:.3f}')
print(f'  Has-result queries:  {refine_rate[0]:.3f}')
print(f'\nUsers refine {refine_rate[1]/refine_rate[0]:.1f}× more often after zero results')

fig, ax = plt.subplots(figsize=(6, 4))
refine_rate.index = ['Has Results', 'Zero Results']
refine_rate.plot.bar(ax=ax, color=['#4CAF50', '#F44336'], edgecolor='black')
ax.set_title('Query Refinement Rate')
ax.set_ylabel('Refinement Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'refinement_rate.png'), dpi=100, bbox_inches='tight')
plt.show()

## Search Volume Trends

In [ ]:
daily_vol = df.groupby('date').size()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_vol.index, daily_vol.values, alpha=0.5, linewidth=0.8)
ax.plot(daily_vol.index, daily_vol.rolling(7).mean(), color='red', linewidth=1.5, label='7-day MA')
ax.set_title('Daily Search Volume')
ax.set_ylabel('Searches')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'search_volume.png'), dpi=100, bbox_inches='tight')
plt.show()

## Device-Based Search Behavior

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
device_stats = df.groupby('device').agg(
    searches=('query', 'count'),
    avg_results=('results_count', 'mean'),
    ctr=('clicked', 'mean'),
    zero_rate=('zero_result', 'mean')
).round(3)
print(device_stats)

device_stats[['ctr', 'zero_rate']].plot.bar(ax=ax, edgecolor='black')
ax.set_title('CTR and Zero-Result Rate by Device')
ax.set_ylabel('Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'device_search.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Long-tail**: Top 10 queries dominate volume — focus search optimization on these first
- **Zero-result queries** are mostly misspellings (`labtop`, `iphoen`, `blutooth`) — implementing fuzzy matching / autocomplete would recover these searches
- **`nike air max 2025`** appears as zero-result — indicates product demand that the catalog doesn't satisfy
- **Refinement rate** is 2.5× higher for zero-result queries — users try again but many leave
- **Mobile CTR** is slightly lower — mobile search UX may need improvement
- **Fixing misspelled zero-result queries alone** could recover ~4% of all searches

## Limitations
- Synthetic query distribution — real distributions are more complex
- No session context (what users did before/after search)
- No position-based click analysis (first result vs later results)
- No revenue attribution to search
- Intent classification is simplified

## How to Improve This Project
- Add search result ranking analysis (NDCG, MRR)
- Include autocomplete suggestion effectiveness
- Add session-level analysis (search → browse → convert path)
- Build a query clustering model for automatic intent detection
- Add A/B testing of search ranking algorithms

## Production Considerations
- Real-time zero-result alerting for new content gaps
- Automated synonym/spell-check suggestions
- Search ranking model retraining pipeline
- Query-to-product mapping for merchandising
- Privacy-preserving query analytics (aggregation, anonymization)

## Common Mistakes
- Analyzing search queries without normalizing case and whitespace
- Measuring search success by result count instead of click-through
- Ignoring zero-result queries as noise instead of treating them as signals
- Not distinguishing navigational vs transactional intent when measuring CTR
- Reporting average CTR without accounting for query-volume distribution

## Mini Challenge / Exercises
1. Group misspelled queries with their corrected versions — what is the combined volume?
2. Calculate the estimated revenue loss from zero-result queries (assume $5 avg order value per converted search).
3. What day of the week has the highest search volume?
4. Build a simple fuzzy-match function that maps `labtop` → `laptop` and measure recovered searches.

## Final Summary / Key Takeaways
- Search logs are a goldmine of user intent data — more direct than browsing behavior
- Zero-result queries reveal both content gaps and UX gaps (spelling, synonyms)
- Long-tail queries are individually rare but collectively significant
- Device-specific search behavior requires device-specific UX optimization
- Fixing the top 5 zero-result terms can recover hundreds of monthly searches